# **Cell 1**

In [ ]:
import os, json, re, random, pathlib
import numpy as np
import tensorflow as tf
import keras

print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus if gpus else "NONE - use Runtime > Change runtime type > T4 GPU")
keras.utils.set_random_seed(42)

# **Cell 2 : Download PlantVillage from Kaggle**

In [ ]:
!pip -q install kagglehub ai-edge-litert
import kagglehub

SLUG = "abdallahalidev/plantvillage-dataset"
try:
    dataset_path = kagglehub.dataset_download(SLUG)
except Exception as exc:
    print("The download asked for a Kaggle login:", type(exc).__name__)
    kagglehub.login()   # paste your Kaggle username and API key (Kaggle > Settings > API)
    dataset_path = kagglehub.dataset_download(SLUG)

print("Downloaded to:", dataset_path)

# **Cell 3 : Find the Tomato and Maize classes**

In [ ]:
CROP_PREFIXES = {"Tomato___": "Tomato", "Corn_(maize)___": "Maize"}

# find the folders that hold one sub-folder per class
candidates = []
for dirpath, dirnames, _ in os.walk(dataset_path):
    if any(d.startswith(tuple(CROP_PREFIXES)) for d in dirnames):
        candidates.append(dirpath)

# prefer the 'color' version over 'grayscale' and 'segmented'
candidates.sort(key=lambda p: ("color" not in p.lower(),
                               "segmented" in p.lower() or "grayscale" in p.lower(),
                               len(p)))
if not candidates:
    print("No class folders found. Top-level contents:")
    print(os.listdir(dataset_path))
    raise SystemExit("Paste the lines above to Claude")

print("Candidate folders:", *candidates, sep="\n  ")
ROOT = pathlib.Path(candidates[0])
print("\nUsing:", ROOT, "\n")

class_dirs = sorted(d for d in ROOT.iterdir()
                    if d.is_dir() and d.name.startswith(tuple(CROP_PREFIXES)))
IMAGE_EXTS = {".jpg", ".jpeg", ".png"}

files, labels, class_names = [], [], []
for index, d in enumerate(class_dirs):
    images = sorted(p for p in d.iterdir() if p.suffix.lower() in IMAGE_EXTS)
    class_names.append(d.name)
    files += [str(p) for p in images]
    labels += [index] * len(images)
    print(f"{len(images):5d}  {d.name}")

NUM_CLASSES = len(class_names)
print(f"\n{NUM_CLASSES} classes, {len(files)} images")

## **Cell 3 : Split the data and build the input pipeline**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

train_f, rest_f, train_y, rest_y = train_test_split(
    files, labels, test_size=0.30, stratify=labels, random_state=42)
val_f, test_f, val_y, test_y = train_test_split(
    rest_f, rest_y, test_size=0.50, stratify=rest_y, random_state=42)
print("train / validation / test:", len(train_f), len(val_f), len(test_f))

weights = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=np.array(train_y))
class_weight = {i: float(w) for i, w in enumerate(weights)}

IMG, BATCH, AUTOTUNE = 224, 32, tf.data.AUTOTUNE

def load_image(path, label):
    data = tf.io.read_file(path)
    image = tf.io.decode_image(data, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, (IMG, IMG), antialias=True)
    return tf.cast(tf.round(image), tf.uint8), label

def make_dataset(paths, ys, training):
    ds = tf.data.Dataset.from_tensor_slices((paths, ys))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(2048, seed=42, reshuffle_each_iteration=True)
    return ds

# augmentation is used while training only (pixel values are 0-255)
augment = keras.Sequential([
    keras.layers.RandomFlip("horizontal_and_vertical"),
    keras.layers.RandomRotation(0.25),
    keras.layers.RandomZoom((-0.3, 0.1)),
    keras.layers.RandomTranslation(0.15, 0.15),
    keras.layers.RandomBrightness(0.3, value_range=(0, 255)),
    keras.layers.RandomContrast(0.3),
])

def add_noise(images, stddev):
    """Gaussian pixel noise for 0-255 images (Keras' own layer only accepts 0-1)."""
    return images + tf.random.normal(tf.shape(images), stddev=stddev)

def prepare(ds, training):
    ds = ds.batch(BATCH)
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(
            lambda x, y: (tf.clip_by_value(add_noise(augment(x, training=True), 8.0), 0.0, 255.0), y),
            num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds = prepare(make_dataset(train_f, train_y, True), True)
val_ds = prepare(make_dataset(val_f, val_y, False), False)
test_ds = prepare(make_dataset(test_f, test_y, False), False)

In [ ]:
augment = keras.Sequential([
    keras.layers.RandomFlip("horizontal_and_vertical"),
    keras.layers.RandomRotation(0.1),
    keras.layers.RandomZoom((-0.15, 0.0)),
    keras.layers.RandomBrightness(0.15, value_range=(0, 255)),
    keras.layers.RandomContrast(0.15),
])

def prepare_train(ds):
    ds = ds.batch(BATCH)
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (tf.clip_by_value(augment(x, training=True), 0.0, 255.0), y),
                num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds = prepare_train(make_dataset(train_f, train_y, True))
print("training pipeline rebuilt")

# **Cell 4 : Build and train the model**

In [ ]:
def build_model(num_classes):
    base = keras.applications.MobileNetV2(
        input_shape=(IMG, IMG, 3), include_top=False, weights="imagenet")
    base.trainable = False

    inputs = keras.Input(shape=(IMG, IMG, 3), name="image")   # float32, pixel values 0-255
    x = keras.layers.Rescaling(1 / 127.5, offset=-1)(inputs)  # to [-1, 1], as MobileNetV2 expects
    x = base(x, training=False)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(num_classes, activation="softmax", name="probabilities")(x)
    return keras.Model(inputs, outputs), base

model, base = build_model(NUM_CLASSES)

# stage 1: train only the new classification head
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h1 = model.fit(
    train_ds, validation_data=val_ds, epochs=8, class_weight=class_weight,
    callbacks=[keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=3, restore_best_weights=True)])

# stage 2: fine-tune most of the base network (BatchNorm layers stay frozen)
base.trainable = True
for layer in base.layers[:-100]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-4),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h2 = model.fit(
    train_ds, validation_data=val_ds, epochs=20, class_weight=class_weight,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=5, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6),
    ])

print("\nLast epochs of fine-tuning")
for name in ("accuracy", "val_accuracy"):
    print(f"{name}:", [round(v, 3) for v in h2.history[name][-3:]])

# **Cell 5 : Evaluate on the held-out test set**

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt

y_true = np.array(test_y)
test_probs = model.predict(test_ds, verbose=0)
test_pred = test_probs.argmax(axis=1)
print("Test accuracy:", round(accuracy_score(y_true, test_pred), 4), "\n")

short = [n.split("___")[1][:30] + " (" + n.split("___")[0][:6] + ")" for n in class_names]
print(classification_report(y_true, test_pred, target_names=short, digits=3))

cm = confusion_matrix(y_true, test_pred)
fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(cm, cmap="Greens")
ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(range(NUM_CLASSES))
ax.set_yticklabels([f"{i}: {s}" for i, s in enumerate(short)], fontsize=8)
ax.set_xlabel("Predicted class number"); ax.set_ylabel("True class")
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7)
plt.tight_layout(); plt.show()

# **Cell 6 : Stress test with messy images**

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

y_true = np.array(test_y)
test_probs = model.predict(test_ds, verbose=0)
test_pred = test_probs.argmax(axis=1)
print("Test accuracy:", round(accuracy_score(y_true, test_pred), 4), "\n")

short = [n.split("___")[1][:30] + " (" + n.split("___")[0][:6] + ")" for n in class_names]
print(classification_report(y_true, test_pred, target_names=short, digits=3))

cm = confusion_matrix(y_true, test_pred)
pairs = [(cm[i, j], i, j) for i in range(NUM_CLASSES) for j in range(NUM_CLASSES)
         if i != j and cm[i, j]]
print("Most common mix-ups (true -> predicted):")
for count, i, j in sorted(pairs, reverse=True)[:8]:
    print(f"  {count:3d}  {short[i]}  ->  {short[j]}")

# **Cell 7 : How confidence relates to being right**

In [ ]:
phone = keras.Sequential([
    keras.layers.RandomZoom((-0.15, -0.05)),
    keras.layers.RandomRotation(0.08),
    keras.layers.RandomBrightness(0.2, value_range=(0, 255)),
    keras.layers.RandomContrast(0.2),
])

def phone_like(x):
    x = phone(x, training=True)
    x = tf.nn.avg_pool2d(x, ksize=3, strides=1, padding="SAME")          # slight blur
    x = x * tf.random.uniform([tf.shape(x)[0], 1, 1, 3], 0.92, 1.08)      # slight colour cast
    x = add_noise(x, 8.0)                                                  # sensor noise
    return tf.clip_by_value(x, 0.0, 255.0)

def degraded_probs():
    return np.concatenate([model.predict_on_batch(phone_like(x)) for x, _ in test_ds])

runs = [degraded_probs() for _ in range(3)]
accs = [accuracy_score(y_true, p.argmax(axis=1)) for p in runs]
print("Accuracy on phone-like degraded images (3 runs):",
      [round(a, 3) for a in accs], "| mean", round(float(np.mean(accs)), 3))
degraded = runs[0]

def bucket_table(probs, title):
    confidence = probs.max(axis=1)
    correct = probs.argmax(axis=1) == y_true
    print(f"\n{title}")
    print("confidence   images   accuracy")
    edges = [0.0, 0.5, 0.7, 0.8, 0.9, 0.95, 1.01]
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (confidence >= lo) & (confidence < hi)
        if mask.sum():
            print(f"{lo:.2f}-{min(hi, 1.0):.2f}   {mask.sum():6d}   {correct[mask].mean():.3f}")

bucket_table(test_probs, "Plain test images")
bucket_table(degraded, "Phone-like degraded images")

# **Cell 8 : Export to TensorFlow Lite and verify with LiteRT**

In [ ]:
os.makedirs("export", exist_ok=True)
model.export("export/saved_model")

converter = tf.lite.TFLiteConverter.from_saved_model("export/saved_model")
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_bytes = converter.convert()
with open("disease_model.tflite", "wb") as f:
    f.write(tflite_bytes)
print("Model size: %.1f MB" % (len(tflite_bytes) / 1e6))

# --- check it with the same runtime and the same preprocessing the backend will use ---
from ai_edge_litert.interpreter import Interpreter
from PIL import Image, ImageOps

def preprocess(pil_image):
    image = ImageOps.exif_transpose(pil_image).convert("RGB")
    side = min(image.size)
    left, top = (image.width - side) // 2, (image.height - side) // 2
    image = image.crop((left, top, left + side, top + side))
    image = image.resize((IMG, IMG), Image.Resampling.BILINEAR)
    return np.asarray(image, dtype=np.float32)[None, ...]

interpreter = Interpreter(model_path="disease_model.tflite")
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]
print("Input:", inp["shape"], inp["dtype"], "| Output:", out["shape"], out["dtype"])

def lite_predict(array):
    interpreter.set_tensor(inp["index"], array)
    interpreter.invoke()
    return interpreter.get_tensor(out["index"])[0]

sample = random.Random(0).sample(range(len(test_f)), 200)
agree = correct_lite = 0
for i in sample:
    array = preprocess(Image.open(test_f[i]))
    lite = lite_predict(array)
    keras_probs = model.predict(array, verbose=0)[0]
    agree += int(lite.argmax() == keras_probs.argmax())
    correct_lite += int(lite.argmax() == test_y[i])
print(f"LiteRT agrees with Keras on {agree}/200 images; "
      f"LiteRT accuracy on them: {correct_lite / 200:.3f}")

# **Cell 9 : Write the labels file and download both files**

In [ ]:
def clean(text):
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")

CROP_NAMES = {"Tomato": "Tomato", "Corn_(maize)": "Maize"}
entries = []
for index, name in enumerate(class_names):
    raw_crop, raw_condition = name.split("___", 1)
    key = clean(raw_condition)
    entries.append({
        "index": index,
        "label": name,
        "crop": CROP_NAMES[raw_crop],
        "key": key,
        "healthy": key == "healthy",
    })

meta = {
    "image_size": IMG,
    "input": "float32 RGB, pixel values 0-255 (the model rescales internally)",
    "classes": entries,
}
with open("labels.json", "w") as f:
    json.dump(meta, f, indent=2)

for e in entries:
    print(e["index"], "|", e["crop"], "|", e["key"])

from google.colab import files
files.download("disease_model.tflite")
files.download("labels.json")

In [ ]:
model.save("model_clean.keras")     # keep the current model safe

def layer_test(layer):
    return lambda x: layer(x, training=True)

def blur5(x):
    return tf.nn.avg_pool2d(x, ksize=5, strides=1, padding="SAME")

TESTS = {
    "no change": lambda x: x,
    "brightness +-30%": layer_test(keras.layers.RandomBrightness(0.3, value_range=(0, 255))),
    "contrast +-30%": layer_test(keras.layers.RandomContrast(0.3)),
    "warm colour cast": lambda x: x * tf.constant([1.15, 1.0, 0.85]),
    "cool colour cast": lambda x: x * tf.constant([0.85, 1.0, 1.15]),
    "blur (5x5)": blur5,
    "noise sigma 10": lambda x: add_noise(x, 10.0),
    "noise sigma 20": lambda x: add_noise(x, 20.0),
    "rotation up to 36 deg": layer_test(keras.layers.RandomRotation(0.1)),
    "rotation up to 90 deg": layer_test(keras.layers.RandomRotation(0.25)),
    "zoom in 30%": layer_test(keras.layers.RandomZoom((-0.30, -0.25))),
    "zoom in 50%": layer_test(keras.layers.RandomZoom((-0.50, -0.45))),
    "shift up to 20%": layer_test(keras.layers.RandomTranslation(0.2, 0.2)),
}

def run_diagnostics(title):
    print(f"\n{title}")
    for name, transform in TESTS.items():
        preds = []
        for x, _ in test_ds:
            x = tf.clip_by_value(transform(x), 0.0, 255.0)
            preds.append(model.predict_on_batch(x).argmax(axis=1))
        acc = accuracy_score(y_true, np.concatenate(preds))
        print(f"  {name:24s} {acc:.3f}")

run_diagnostics("Current model (light augmentation)")

In [ ]:
# --- moderate augmentation, between the two earlier settings ---
augment_v3 = keras.Sequential([
    keras.layers.RandomFlip("horizontal_and_vertical"),
    keras.layers.RandomRotation(0.15),
    keras.layers.RandomZoom((-0.25, 0.1)),
    keras.layers.RandomTranslation(0.1, 0.1),
    keras.layers.RandomBrightness(0.25, value_range=(0, 255)),
    keras.layers.RandomContrast(0.25),
])

def color_jitter(x, y):
    x = x / 255.0                                  # the hsv-based ops expect 0-1
    x = tf.image.random_saturation(x, 0.7, 1.3)
    x = tf.image.random_hue(x, 0.04)
    return tf.clip_by_value(x, 0.0, 1.0) * 255.0, y

def prepare_train_v3(ds):
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    ds = ds.map(color_jitter, num_parallel_calls=AUTOTUNE)      # per image, before batching
    ds = ds.batch(BATCH)
    ds = ds.map(
        lambda x, y: (tf.clip_by_value(add_noise(augment_v3(x, training=True), 5.0), 0.0, 255.0), y),
        num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds = prepare_train_v3(make_dataset(train_f, train_y, True))

# --- build and train again from scratch ---
model, base = build_model(NUM_CLASSES)

model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(train_ds, validation_data=val_ds, epochs=8, class_weight=class_weight,
          callbacks=[keras.callbacks.EarlyStopping(
              monitor="val_accuracy", patience=3, restore_best_weights=True)])

base.trainable = True
for layer in base.layers[:-100]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-4),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h2 = model.fit(train_ds, validation_data=val_ds, epochs=30, class_weight=class_weight,
               callbacks=[
                   keras.callbacks.EarlyStopping(
                       monitor="val_accuracy", patience=6, restore_best_weights=True),
                   keras.callbacks.ReduceLROnPlateau(
                       monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6),
               ])

print("\nLast epochs of fine-tuning")
for name in ("accuracy", "val_accuracy"):
    print(f"{name}:", [round(v, 3) for v in h2.history[name][-3:]])

model.save("model_moderate.keras")

In [ ]:
y_true = np.array(test_y)
test_probs = model.predict(test_ds, verbose=0)
print("Test accuracy:", round(accuracy_score(y_true, test_probs.argmax(axis=1)), 4))
run_diagnostics("New model (moderate augmentation)")